In [ ]:
import pandas as pd

In [ ]:
# ===============================
# 1. LOAD LIBRARIES
# ===============================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm


In [ ]:
# ===============================
# 2. LOAD DATA
# ===============================
df = pd.read_csv("HIV_Patients.csv")

In [ ]:

# ===============================
# 3. DROP NON-ANALYTICAL COLUMNS
# ===============================
df.drop(columns=["Patient_ID", "Date_Of_Diagnosis"], inplace=True)


In [ ]:
# ===============================
# 4. CREATE BINARY OUTCOME
# Viral Suppression: <1000 copies/ml
# ===============================
df["Viral_Suppressed"] = (df["Viral_Load"] < 1000).astype(int)
df.drop(columns=["Viral_Load"], inplace=True)


In [ ]:
# ===============================
# 5. IDENTIFY VARIABLE TYPES
# ===============================
categorical_cols = df.select_dtypes(include="object").columns
continuous_cols = df.select_dtypes(exclude="object").drop("Viral_Suppressed", axis=1).columns

In [ ]:

# ===============================
# 6. HANDLE MISSING VALUES
# ===============================
for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

for col in continuous_cols:
    df[col].fillna(df[col].mean(), inplace=True)


In [ ]:
# ===============================
# 7. TARGET ENCODING (ROBUST)
# ===============================
for col in categorical_cols:
    mean_map = df.groupby(col)["Viral_Suppressed"].mean()
    df[col] = df[col].map(mean_map)


In [ ]:
# ===============================
# 8. FEATURE SCALING
# ===============================
scaler = StandardScaler()
df[continuous_cols] = scaler.fit_transform(df[continuous_cols])


In [ ]:
# ===============================
# 9. TRAIN-TEST SPLIT
# ===============================
X = df.drop("Viral_Suppressed", axis=1)
y = df["Viral_Suppressed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [ ]:
# ===============================
# 10. LOGISTIC REGRESSION (INFERENCE)
# ===============================
X_train_sm = sm.add_constant(X_train)
model = sm.Logit(y_train, X_train_sm).fit(disp=False)


In [ ]:

# ===============================
# 11. ODDS RATIOS & STATISTICAL OUTPUT
# ===============================
results = pd.DataFrame({
    "Coefficient": model.params,
    "Std_Error": model.bse,
    "Z_value": model.tvalues,
    "P_value": model.pvalues,
    "Odds_Ratio": np.exp(model.params),
    "CI_2.5%": np.exp(model.conf_int()[0]),
    "CI_97.5%": np.exp(model.conf_int()[1])
})

results = results.drop("const")

print(results.sort_values("P_value"))
